In [1]:
import ifcopenshell
import ifcopenshell.api
import ifcopenshell.util
import ifcopenshell.api.root
import ifcopenshell.api.context
import ifcopenshell.api.aggregate

In [2]:
model = ifcopenshell.open("./blenderbim-site-library.ifc")
model.schema

'IFC4'

In [3]:
project = model.by_type("IfcProject")[0]

# Set up units - using millimeters
units = model.create_entity("IfcUnitAssignment")
length_unit = model.create_entity("IfcSIUnit", UnitType="LENGTHUNIT", Name="METRE")  # Remove the MILLI prefix
units.Units = [length_unit]
project.UnitsInContext = units

# Set up geometric representation contexts
context = model.create_entity(
    "IfcGeometricRepresentationContext",
    ContextType="Model",
    CoordinateSpaceDimension=3,
    Precision=0.01,
    WorldCoordinateSystem=model.create_entity(
        "IfcAxis2Placement3D",
        Location=model.create_entity("IfcCartesianPoint", Coordinates=(0.0, 0.0, 0.0)),
    ),
)

model_context = model.create_entity(
    "IfcGeometricRepresentationSubContext",
    ContextIdentifier="Body",
    ContextType="Model",
    ParentContext=context,
    TargetView="MODEL_VIEW",
)
# context = ifcopenshell.api.context.add_context(model, context_type="Model")
# body = ifcopenshell.api.context.add_context(model, context_type="Model",
#     context_identifier="Body", target_view="MODEL_VIEW", parent=context)


# Create a site, building, and storey. Many hierarchies are possible.
site = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSite", name="My Site")
building = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuilding", name="Building A")
storey = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuildingStorey", name="Ground Floor")

ifcopenshell.api.aggregate.assign_object(model, relating_object=project, products=[site])
ifcopenshell.api.aggregate.assign_object(model, relating_object=site, products=[building])
ifcopenshell.api.aggregate.assign_object(model, relating_object=building, products=[storey])

#314=IfcRelAggregates('1sPfdusbX1QOIRrMIWrOni',$,$,$,#310,(#311))

In [4]:
lib = model.by_type("IfcProjectLibrary")[0]
lib

#8=IfcProjectLibrary('3ukNlWeMfCufxIIv6a2tjG',$,'BlenderBIM Demo Library',$,$,$,$,$,$)

In [5]:
# Get the contents of the lib
library_objects = lib.Declares[0].RelatedDefinitions
for obj in library_objects:
    print(obj)

#91=IfcBuildingElementProxyType('0FMiZScTPFog7h7dFJ1g95',$,'Mobile Crane 50T',$,$,$,(#227,#302),$,$,$)
#27=IfcBuildingElementProxyType('2YQNXdYf18kBzxu3tjWlgr',$,'Site Shed 3x6m',$,$,$,(#58),$,$,$)
#59=IfcBuildingElementProxyType('0Q0hvi_v97dhIU2pPJfAJ5',$,'Site Shed 3x12m',$,$,$,(#90),$,$,$)
#8=IfcProjectLibrary('3ukNlWeMfCufxIIv6a2tjG',$,'BlenderBIM Demo Library',$,$,$,$,$,$)


In [17]:
crane_type = library_objects[0]

In [7]:
import ifcopenshell.api.geometry
import ifcopenshell.api.spatial


crane = model.create_entity("IfcBuildingElementProxy", Name="cool crane")
crane.ObjectType = crane_type.Name
# ifcopenshell.api.geometry.edit_object_placement(model, product=crane)
# ifcopenshell.api.spatial.assign_container(model, relating_structure=storey, products=[crane])

In [8]:
type_relationship = model.create_entity(
    "IfcRelDefinesByType",
    GlobalId=ifcopenshell.guid.new(),
    RelatedObjects=[crane],
    RelatingType=crane_type
)

In [9]:
# Create placement for the crane
placement = model.create_entity(
    "IfcLocalPlacement",
    PlacementRelTo=storey.ObjectPlacement,
    RelativePlacement=model.create_entity(
        "IfcAxis2Placement3D",
        Location=model.create_entity(
            "IfcCartesianPoint",
            Coordinates=(0.0, 0.0, 0.0)
        )
    )
)
crane.ObjectPlacement = placement

In [10]:
if crane_type.RepresentationMaps:
    context = model_context
    shape = model.create_entity(
        "IfcShapeRepresentation",
        ContextOfItems=context,
        RepresentationIdentifier=crane_type.RepresentationMaps[0].MappedRepresentation.RepresentationIdentifier,
        RepresentationType=crane_type.RepresentationMaps[0].MappedRepresentation.RepresentationType,
    )
    
    # Create mapping
    mapped_item = model.create_entity(
        "IfcMappedItem",
        MappingSource=crane_type.RepresentationMaps[0],
        MappingTarget=model.create_entity(
            "IfcCartesianTransformationOperator3D",
            Axis1=None,
            Axis2=None,
            LocalOrigin=model.create_entity(
                "IfcCartesianPoint",
                Coordinates=(0.0, 0.0, 0.0)
            ),
            Scale=1.0,
            Axis3=None
        )
    )
    shape.Items = [mapped_item]
    
    # Create product definition shape
    product_shape = model.create_entity(
        "IfcProductDefinitionShape",
        Representations=[shape]
    )
    crane.Representation = product_shape

# Assign the crane to the storey
ifcopenshell.api.aggregate.assign_object(
    model,
    relating_object=storey,
    products=[crane]
)


#325=IfcRelAggregates('1r5tvIbmz1zvvvMU_GVZVi',$,$,$,#311,(#315))

In [11]:
model.write("test.ifc")